# Fine-tuned arm: ResNet-18 stroke classifier

**Model category: fine-tuned** (ImageNet-pretrained ResNet-18, new 5-class head).
This is the second model category the course requires, next to Serena's trained-from-scratch arm in `sketch2stl/recognizer/ml.py`.

What it does: draw each stroke as a small image, then classify it as `line / arc / circle / rect / polyline` (same classes as `types.PrimitiveKind`).
Like the other arms it only predicts the **class**; the geometry (centre, radius…) still comes from `recognizer/rules.py` fitters (decision D4).

Data:
- **Train / val:** synthetic strokes generated from Fusion 360 sketches (`tools/fusion_strokes.py`). Labels come from the CAD history. Split by Fusion project.
- **Test that counts:** our hand-drawn strokes from `tools/stroke_collector.html` (`data/strokes/raw/*.json`). Never trained on.

Runtime: **Runtime → Change runtime type → T4 GPU.** Training takes ~5–10 minutes.

In [ ]:
# ---- settings -------------------------------------------------------------
DATASET_REPO = "pakiino/sketch2stl-strokes"   # HF dataset holding fusion_strokes.npz, labels.csv and raw/*.json
MODEL_REPO   = "pakiino/sketch2stl-resnet18"  # where to push the trained model (optional)
IMG = 64          # stroke raster size
EPOCHS = 6
BATCH = 256
LR = 1e-3
SEED = 20260923
QUICK_TEST = False   # True = tiny run on CPU to check the notebook works

In [ ]:
import os, glob, json, csv, random, time
import numpy as np, cv2, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["line", "arc", "circle", "rect", "polyline"]   # same order as recognizer/ml.py
print("device:", DEVICE)

## 1. Get the data
Downloads from the Hugging Face dataset. If you are running locally instead, set `DATA_DIR` to your `data/strokes` folder.

In [ ]:
DATA_DIR = os.environ.get("DATA_DIR", "")
if not DATA_DIR:
    from huggingface_hub import snapshot_download
    DATA_DIR = snapshot_download(repo_id=DATASET_REPO, repo_type="dataset")
print(DATA_DIR, os.listdir(DATA_DIR))

In [ ]:
def load_synthetic(folder):
    z = np.load(os.path.join(folder, "fusion_strokes.npz")); P, O = z["points"], z["offsets"]
    rows = list(csv.DictReader(open(os.path.join(folder, "labels.csv"))))
    return [P[O[i]:O[i+1]].astype(np.float32) for i in range(len(rows))], rows

def load_hand(folder):
    strokes, rows = [], []
    for f in sorted(glob.glob(os.path.join(folder, "*.json"))):
        s = json.load(open(f))
        for st in s["strokes"]:
            p = np.asarray(st["points"], np.float32).reshape(-1, 2)
            if len(p) >= 4:
                strokes.append(p); rows.append({"label": st["label"], "session_id": s["session_id"], "author": s["author"]})
    return strokes, rows

syn_dir = DATA_DIR if os.path.exists(os.path.join(DATA_DIR, "fusion_strokes.npz")) else os.path.join(DATA_DIR, "synthetic")
S, R = load_synthetic(syn_dir)
H, HR = load_hand(os.path.join(DATA_DIR, "raw"))
print(f"synthetic strokes: {len(S)}   hand-drawn strokes: {len(H)}")

def pick(split):
    idx = [i for i, r in enumerate(R) if r["split"] == split]
    if QUICK_TEST: idx = idx[:600]
    return [S[i] for i in idx], [CLASSES.index(R[i]["label"]) for i in idx]
Xtr, ytr = pick("train"); Xva, yva = pick("val"); Xte, yte = pick("test")
print("train", len(Xtr), "val", len(Xva), "synthetic test", len(Xte))

## 2. Strokes → images
Centre the stroke, scale it to fit (keeping the aspect ratio so a line stays thin), and draw it. Training adds small random rotations, stretch and line thickness so the model does not memorise one drawing style.

In [ ]:
def rasterize(pts, size=IMG, thickness=2, aug=False, rng=np.random):
    pts = np.asarray(pts, np.float64)
    if aug:
        a = rng.uniform(-0.25, 0.25); c, s = np.cos(a), np.sin(a)
        pts = pts @ np.array([[c, -s], [s, c]]).T * np.array([rng.uniform(0.85, 1.15), rng.uniform(0.85, 1.15)])
        thickness = rng.choice([1, 2, 2, 3])
    img = np.zeros((size, size), np.uint8)
    lo, hi = pts.min(0), pts.max(0)
    q = (pts - (lo + hi) / 2) * ((size - 8) / max((hi - lo).max(), 1e-6)) + size / 2
    cv2.polylines(img, [np.round(q).astype(np.int32)], False, 255, int(thickness), cv2.LINE_AA)
    return img

class StrokeSet(Dataset):
    def __init__(self, X, y, aug=False): self.X, self.y, self.aug = X, y, aug
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        img = torch.from_numpy(rasterize(self.X[i], aug=self.aug)).float().div(255)
        img = (img - 0.5) / 0.5
        return img.unsqueeze(0).repeat(3, 1, 1), self.y[i]      # 3 channels for the ImageNet backbone

dl = lambda X, y, aug, sh: DataLoader(StrokeSet(X, y, aug), batch_size=BATCH, shuffle=sh, num_workers=2)
train_dl, val_dl, test_dl = dl(Xtr, ytr, True, True), dl(Xva, yva, False, False), dl(Xte, yte, False, False)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 10, figsize=(15, 2))
for k in range(10):
    i = random.randrange(len(Xtr)); ax[k].imshow(rasterize(Xtr[i], aug=True), cmap="gray"); ax[k].set_title(CLASSES[ytr[i]]); ax[k].axis("off")

## 3. Fine-tune ResNet-18
Start from ImageNet weights, replace the last layer with a 5-class head. All layers are fine-tuned with a small learning rate on the backbone and a larger one on the new head.

In [ ]:
try:
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    model = torchvision.models.resnet18(weights=weights)
except Exception as e:                      # offline: fall back so the notebook still runs (NOT fine-tuned then)
    print("could not download ImageNet weights:", e); model = torchvision.models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(DEVICE)
head = list(model.fc.parameters()); body = [p for n, p in model.named_parameters() if not n.startswith("fc.")]
opt = torch.optim.AdamW([{"params": body, "lr": LR * 0.1}, {"params": head, "lr": LR}], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR * 0.1, LR], total_steps=EPOCHS * len(train_dl))
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

def evaluate(loader):
    model.eval(); P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            P.append(model(x.to(DEVICE)).argmax(1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

best, history = 0, []
for ep in range(EPOCHS if not QUICK_TEST else 1):
    model.train(); t0 = time.time(); tot = 0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(y)
    p, yv = evaluate(val_dl); acc = (p == yv).mean()
    history.append((ep, tot / len(Xtr), acc)); print(f"epoch {ep}  loss {tot/len(Xtr):.3f}  val acc {acc:.3f}  ({time.time()-t0:.0f}s)")
    if acc >= best: best = acc; torch.save(model.state_dict(), "resnet18_strokes.pt")
model.load_state_dict(torch.load("resnet18_strokes.pt"))

## 4. Evaluate
Report **per-class** accuracy, not just overall (the repo's rule: "94% overall usually hides 0% on arcs"). The hand-drawn set is the number that goes in the report; the synthetic test only shows whether training worked.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
def report(name, X, y):
    if not X: print(f"{name}: no strokes yet"); return None
    p, yy = evaluate(dl(X, y, False, False))
    print(f"\n=== {name}: accuracy {np.mean(p == yy):.3f} on {len(yy)} strokes")
    print(classification_report(yy, p, labels=range(len(CLASSES)), target_names=CLASSES, zero_division=0))
    cm = confusion_matrix(yy, p, labels=range(len(CLASSES)))
    fig, ax = plt.subplots(figsize=(4.5, 4)); ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(5), CLASSES, rotation=45); ax.set_yticks(range(5), CLASSES); ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name)
    for i in range(5):
        for j in range(5): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    plt.tight_layout(); plt.show(); return p

report("synthetic test (Fusion)", Xte, yte)
hand_pred = report("HAND-DRAWN test (the real test)", H, [CLASSES.index(r["label"]) for r in HR])

## 5. Save and share
Saves the weights with the class order. `recognizer/ml.py` can load this file and call the matching fitter from `rules.py`.

In [ ]:
torch.save({"state_dict": model.state_dict(), "classes": CLASSES, "img": IMG, "arch": "resnet18"}, "resnet18_strokes_full.pt")
json.dump({"val_acc_best": float(best), "history": history}, open("training_log.json", "w"), indent=1)
PUSH = False   # set True after `from huggingface_hub import notebook_login; notebook_login()`
if PUSH:
    from huggingface_hub import HfApi
    api = HfApi(); api.create_repo(MODEL_REPO, exist_ok=True)
    for f in ["resnet18_strokes_full.pt", "training_log.json"]: api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=MODEL_REPO)
    print("pushed to", MODEL_REPO)